In [3]:
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler

def generar_dataset(num_muestras=50000, rango=(-20, 20)):
    """
    Paso 1: Generación del dataset.
    Crea 'num_muestras' de pares de matrices 2x2 con enteros aleatorios y calcula su producto.
    """
    print(f"--- Generando dataset de {num_muestras} multiplicaciones ---")
    A = np.random.randint(rango[0], rango[1] + 1, size=(num_muestras, 2, 2))
    B = np.random.randint(rango[0], rango[1] + 1, size=(num_muestras, 2, 2))

    C = np.matmul(A, B)

    X = np.hstack((A.reshape(num_muestras, 4), B.reshape(num_muestras, 4)))
    y = C.reshape(num_muestras, 4)

    print("Dataset generado con éxito.\n")
    return X, y

def entrenar_modelo(X, y):
    """
    Paso 2: Entrenamiento del modelo de Machine Learning con escalado y parada temprana.
    """
    print("--- Dividiendo datos y entrenando el modelo ---")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Normalizamos los datos (Media 0, Varianza 1)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Red neuronal con Early Stopping
    modelo = MLPRegressor(hidden_layer_sizes=(128, 128),
                          max_iter=500,
                          early_stopping=True,
                          random_state=42)

    inicio_entrenamiento = time.time()
    modelo.fit(X_train_scaled, y_train)
    fin_entrenamiento = time.time()

    predicciones = modelo.predict(X_test_scaled)
    mse = mean_squared_error(y_test, predicciones)

    print(f"Modelo entrenado en {fin_entrenamiento - inicio_entrenamiento:.2f} segundos.")
    print(f"Épocas reales utilizadas: {modelo.n_iter_}")
    print(f"Error Cuadrático Medio (MSE) en test: {mse:.2f}\n")

    return modelo, scaler # Retornamos ambos objetos

def evaluar_y_comparar_costo(modelo, scaler, num_ejemplos=10):
    """
    Paso 3: Evaluación. IMPORTANTE: Los datos nuevos deben ser escalados.
    """
    print(f"--- Evaluación con {num_ejemplos} ejemplos nuevos ---")
    A_nuevas = np.random.randint(-20, 21, size=(num_ejemplos, 2, 2))
    B_nuevas = np.random.randint(-20, 21, size=(num_ejemplos, 2, 2))

    X_nuevo = np.hstack((A_nuevas.reshape(num_ejemplos, 4), B_nuevas.reshape(num_ejemplos, 4)))

    # CORRECCIÓN: Escalar los datos nuevos antes de predecir
    X_nuevo_scaled = scaler.transform(X_nuevo)

    inicio_ml = time.perf_counter()
    predicciones_ml = modelo.predict(X_nuevo_scaled)
    fin_ml = time.perf_counter()
    tiempo_ml = fin_ml - inicio_ml

    inicio_analitico = time.perf_counter()
    resultados_analiticos = np.matmul(A_nuevas, B_nuevas).reshape(num_ejemplos, 4)
    fin_analitico = time.perf_counter()
    tiempo_analitico = fin_analitico - inicio_analitico

    print("Ejemplo 1:")
    print(f"Matriz A:\n{A_nuevas[0]}")
    print(f"Matriz B:\n{B_nuevas[0]}")
    print(f"Resultado Analítico Real:\n{resultados_analiticos[0].reshape(2,2)}")
    print(f"Predicción del Modelo ML (redondeada):\n{np.round(predicciones_ml[0]).reshape(2,2)}")

    print("\n--- Análisis de Costo Computacional ---")
    print(f"Tiempo total analítico ({num_ejemplos} ejemplos): {tiempo_analitico:.6f} segundos")
    print(f"Tiempo total modelo ML ({num_ejemplos} ejemplos): {tiempo_ml:.6f} segundos")

    if tiempo_ml > tiempo_analitico:
        print("Conclusión: El modelo de Machine Learning es MÁS COSTOSO computacionalmente.")
    else:
        print("Conclusión: El método analítico es MÁS COSTOSO computacionalmente.")
    print("---------------------------------------\n")

def modo_interactivo(modelo, scaler):
    """
    Paso 4: Función interactiva.
    """
    print("--- Modo Interactivo ---")
    print("Ingresa los valores para multiplicar dos matrices 2x2. (Escribe 'salir' para terminar)")

    while True:
        entrada = input("\n¿Deseas probar una multiplicación generada aleatoriamente? (s/salir): ")
        if entrada.lower() == 'salir':
            break
        elif entrada.lower() != 's':
            continue

        try:
            A = np.random.randint(-10, 10, size=(2, 2))
            B = np.random.randint(-10, 10, size=(2, 2))

            X_input = np.hstack((A.reshape(1, 4), B.reshape(1, 4)))

            # CORRECCIÓN: Escalar el dato interactivo antes de predecir
            X_input_scaled = scaler.transform(X_input)

            t0 = time.perf_counter()
            pred_ml = modelo.predict(X_input_scaled).reshape(2, 2)
            t_ml = time.perf_counter() - t0

            t0 = time.perf_counter()
            res_real = np.matmul(A, B)
            t_real = time.perf_counter() - t0

            print(f"\nMatriz A:\n{A}\nMatriz B:\n{B}")
            print(f"\nResultado Real (tomó {t_real:.6f}s):\n{res_real}")
            print(f"Predicción ML  (tomó {t_ml:.6f}s):\n{np.round(pred_ml, 2)}")

            diferencia = np.abs(res_real - pred_ml)
            print(f"Error absoluto promedio en esta predicción: {np.mean(diferencia):.2f}")

        except Exception as e:
            print(f"Error: {e}")

# --- EJECUCIÓN DEL SCRIPT ---
if __name__ == "__main__":
    # 1. Generar datos
    X, y = generar_dataset(50000)

    # 2. Entrenar modelo (Ahora desempaquetamos correctamente la tupla)
    modelo_entrenado, scaler_entrenado = entrenar_modelo(X, y)

    # 3. Evaluar y comparar (Pasamos el modelo y el scaler)
    evaluar_y_comparar_costo(modelo_entrenado, scaler_entrenado, num_ejemplos=10)

    # 4. Iniciar modo interactivo (Pasamos el modelo y el scaler)
    modo_interactivo(modelo_entrenado, scaler_entrenado)

--- Generando dataset de 50000 multiplicaciones ---
Dataset generado con éxito.

--- Dividiendo datos y entrenando el modelo ---
Modelo entrenado en 58.89 segundos.
Épocas reales utilizadas: 82
Error Cuadrático Medio (MSE) en test: 127.43

--- Evaluación con 10 ejemplos nuevos ---
Ejemplo 1:
Matriz A:
[[  4 -20]
 [ 13  17]]
Matriz B:
[[-15 -16]
 [ -1  11]]
Resultado Analítico Real:
[[ -40 -284]
 [-212  -21]]
Predicción del Modelo ML (redondeada):
[[ -46. -296.]
 [-212.  -13.]]

--- Análisis de Costo Computacional ---
Tiempo total analítico (10 ejemplos): 0.000010 segundos
Tiempo total modelo ML (10 ejemplos): 0.000361 segundos
Conclusión: El modelo de Machine Learning es MÁS COSTOSO computacionalmente.
---------------------------------------

--- Modo Interactivo ---
Ingresa los valores para multiplicar dos matrices 2x2. (Escribe 'salir' para terminar)

¿Deseas probar una multiplicación generada aleatoriamente? (s/salir): s

Matriz A:
[[-9  0]
 [-7 -2]]
Matriz B:
[[  2 -10]
 [  9  -3]]